<a href="https://colab.research.google.com/github/priyalpatil31/AML_Priyal_Patil_19/blob/main/EXP_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive.mount('/content/drive')


df = pd.read_excel('/content/drive/MyDrive/Online_Retail_Small.xlsx')

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom
2,536371,22086,PAPER CHAIN KIT 50'S CHRISTMAS,80,2010-12-01 09:00:00,2.55,13748,United Kingdom
3,536373,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:02:00,2.55,17850,United Kingdom
4,536375,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:32:00,2.55,17850,United Kingdom


In [ ]:
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Basic Stats ===")
print(df[["Quantity", "UnitPrice"]].describe())

=== Data Types ===
InvoiceNo               int64
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
dtype: object

=== Missing Values ===
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

=== Basic Stats ===
         Quantity   UnitPrice
count  769.000000  769.000000
mean    16.027308    3.655488
std     36.563665    3.432685
min      1.000000    1.450000
25%      2.000000    1.650000
50%      6.000000    2.550000
75%     12.000000    2.950000
max    500.000000   12.750000


In [ ]:
df.dropna(inplace=True)
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

print("Cleaned Shape:", df.shape)

Cleaned Shape: (769, 8)


In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Group by customer
customer_df = df.groupby("CustomerID").agg(
    TotalOrders   = ("InvoiceNo", "nunique"),
    TotalQuantity = ("Quantity", "sum"),
    TotalRevenue  = ("Revenue", "sum")
).reset_index()

print("=== Customer Data ===")
print(customer_df.head())
print(customer_df.describe())

=== Customer Data ===
   CustomerID  TotalOrders  TotalQuantity  TotalRevenue
0       12747            2             40         67.60
1       12748            9             72        118.06
2       12826            1              6         17.70
3       12838            1             11         25.95
4       12841            2             11         35.85
         CustomerID  TotalOrders  TotalQuantity  TotalRevenue
count    405.000000   405.000000     405.000000    405.000000
mean   15673.538272     1.234568      30.432099     93.920469
std     1570.014303     1.005820      76.796945    367.131966
min    12747.000000     1.000000       1.000000      1.650000
25%    14415.000000     1.000000       6.000000     17.050000
50%    15723.000000     1.000000      12.000000     33.320000
75%    17126.000000     1.000000      25.000000     78.450000
max    18239.000000    17.000000     896.000000   6771.200000


In [ ]:
X = customer_df[["TotalOrders", "TotalQuantity", "TotalRevenue"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling done!")

Scaling done!


In [ ]:
inertia = []
sil_scores = []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))
    print(f"K={k} | Inertia: {km.inertia_:.2f} | Silhouette: {silhouette_score(X_scaled, km.labels_):.4f}")

K=2 | Inertia: 747.31 | Silhouette: 0.9556
K=3 | Inertia: 451.74 | Silhouette: 0.9259
K=4 | Inertia: 219.49 | Silhouette: 0.8741
K=5 | Inertia: 187.42 | Silhouette: 0.8729
K=6 | Inertia: 144.15 | Silhouette: 0.8540
K=7 | Inertia: 89.14 | Silhouette: 0.7162


In [ ]:
best_k = 3

model = KMeans(n_clusters=best_k, random_state=42)
customer_df["Cluster"] = model.fit_predict(X_scaled)

print("=== Cluster Counts ===")
print(customer_df["Cluster"].value_counts())

=== Cluster Counts ===
Cluster
1    401
2      3
0      1
Name: count, dtype: int64


In [ ]:
print("=== Cluster Summary ===")
print(customer_df.groupby("Cluster")[["TotalOrders", "TotalQuantity", "TotalRevenue"]].mean().round(2))

=== Cluster Summary ===
         TotalOrders  TotalQuantity  TotalRevenue
Cluster                                          
0               4.00         896.00       6771.20
1               1.16          26.18         73.38
2              10.33         310.00        614.25


In [ ]:
# ── Overall Accuracy ─────────────────────────────────────────────
from sklearn.metrics import silhouette_score

accuracy = silhouette_score(X_scaled, customer_df["Cluster"])
print(f"Overall Model Accuracy: {accuracy*100:.2f}%")

Overall Model Accuracy: 92.59%
